# Overview

The objective of this project is to develop solutions based on the design provided. In this case, the source data was obtained in the form of CSV files from a MySQL DB.

To improve the efficiency of our data engineering pipelines, we need to convert these CSV files into JSON files, since `JSON is better to use in downstream applications` than CSV files. The scope of this project involves converting CSV files into JSON files.

In [ ]:
import glob

In [ ]:
help(glob)

In [ ]:
help(glob.glob)

In [ ]:
glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/*')

"If `recursive` is `true`, the pattern '**' will match any files and
zero or more directories and subdirectories."

In [ ]:
glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/**')

In [ ]:
glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/**', recursive=True)

In [ ]:
initial_glob = glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/**', recursive=True)

In [ ]:
type(initial_glob)

All the files in all the folders

In [ ]:
# Listing all the files in the 'retail_db' subdirectories, filtering by the correct pattern and sorting them
src_file_names = sorted(glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/*/part-*')) # Retrieve and sort the file paths alphabetically

In [ ]:
# If the files to be consumed have some common pattern in their names, we can use that pattern to filter the files to be consumed ('part-') - part-00000 is the common pattern in the files we want to consume:
src_file_names = sorted(glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/*/*'))

In [ ]:
src_file_names

In [ ]:
import pandas as pd

In [ ]:
# Understand each file's shape
for file_name in src_file_names:
    df = pd.read_csv(file_name)
    print(f'Shape of {file_name} is {df.shape}')

In [ ]:
import json

In [ ]:
# Creating a function to get column names of a table in the right order
def get_column_names(schemas, table_name, sorting_key):
    column_details = schemas[table_name]
    ## return column_details
    columns = sorted(column_details, key=lambda col: col[sorting_key])
    ##return columns
    return [col['column_name'] for col in columns]

In [ ]:
# Point to schemas json file and get a list of each table and its columns
schemas = json.load(open('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/schemas.json'))

In [ ]:
# Use the function to get the order's column names sorted by sorting_key
order_columns = get_column_names(schemas, 'orders', 'column_position')

In [ ]:
# Column names from orders sorted according to their position in the table
order_columns

In [ ]:
# Creating a pandas dataframe with the right column names (based on the scehmas.json file) and the data from the csv file
orders = pd.read_csv('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/orders/part-00000', names=order_columns)

In [ ]:
# It worked
orders.head()

Creating the Pipeline

In [ ]:
import json
import re
import pandas as pd

# Creating a function to get column names of a table in the right order
def get_column_names(schemas, table_name, sorting_key):
    column_details = schemas[table_name]
    ## return column_details
    columns = sorted(column_details, key=lambda col: col[sorting_key])
    ##return columns
    return [col['column_name'] for col in columns]

# Point to schemas json file and get a list of each table and its columns
schemas = json.load(open('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/schemas.json'))

# Listing all the files in the 'retail_db'subdirectories
glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/*/part*')

file = src_file_names[0]

file_details = re.split(r'\/', file)

table_name = file_details[-2]

columns = get_column_names(schemas, table_name, 'column_position')

df = pd.read_csv(file, names=columns)

df.head() # Devia ser a tabela categories. O erro está em src_file_names, que tem o caminho errado. O caminho certo é: /Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/*/part-*

In [ ]:
for file in src_file_names:
    file_details = re.split(r'\/', file)
    table_name = file_details[-2]
    columns = get_column_names(schemas, table_name, 'column_position')
    df = pd.read_csv(file, names=columns)
    print(f'Shape of {file} is {df.shape}')

In [ ]:
# Last df read was products, so the head of that dataframe is shown below:
df.head()

In [ ]:
tgt_base_dir = '/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db_json'

In [ ]:
for file in src_file_names:
    file_path_list = re.split(r'\/', file)
    print(file_path_list)

In [ ]:
file = src_file_names[0]
file

In [ ]:
file_details = re.split(r'\/', file)
file_details[-2]

In [ ]:
file_name = file_details[-1]
file_name

In [ ]:
json_file_path = f'{tgt_base_dir}/{table_name}/{file_name}'
print(json_file_path)

In [ ]:
import json
import re
import pandas as pd

# If the files to be consumed have some common pattern in their names, we can use that pattern to filter the files to be consumed ('part-') - part-00000 is the common pattern in the files we want to consume:
src_file_names = sorted(glob.glob('/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db/*/*'))

# Creating a function to get column names of a table in the right order
def get_column_names(schemas, table_name, sorting_key):
    column_details = schemas[table_name]
    columns = sorted(column_details, key=lambda col: col[sorting_key])
    return [col['column_name'] for col in columns]


for file in src_file_names:
    file_details = re.split(r'\/', file)
    table_name = file_details[-2]
    file_name = file_details[-1]
    json_file_path = f'{tgt_base_dir}/{table_name}/{file_name}'
    columns = get_column_names(schemas, table_name, 'column_position')
    df = pd.read_csv(file, names=columns)
    os.makedirs(f'{tgt_base_dir}/{table_name}', exist_ok=True) # Create the target directory if it doesn't exist
    df.to_json(json_file_path, orient='records', lines=True) # Write the dataframe to a json file in the target directory
    print(json_file_path)

Monularize the application or code means dividing the overall application logic into manageable chuncks. One of the ways of doint it is by defining the functions depending upon the functionality which we want to achieve using those functions. 

In [87]:
import os
import json
import re
import pandas as pd

In [88]:
# Get the metada (column names)
def get_column_names(schemas, table_name, sorting_key):
    column_details = schemas[table_name]
    columns = sorted(column_details, key=lambda col: col[sorting_key])
    return [col['column_name'] for col in columns]

In [89]:
# Read the data from the source (csv) files and turn them into dfs applying the schema selected on previous function
def read_csv(file,schemas):    
    file_path_list = re.split(r'\/', file)
    table_name = file_path_list[-2]
    file_name = file_path_list[-1]
    columns = get_column_names(schemas, table_name, 'column_position')
    df = pd.read_csv(file, names=columns)
    return df

In [90]:
# Converting the dataframes into json files and writing them to the target directory
def to_json(df, tgt_base_dir, table_name, file_name):
    json_file_path = f'{tgt_base_dir}/{table_name}/{file_name}'
    os.makedirs(f'{tgt_base_dir}/{table_name}', exist_ok=True) # Create the target directory if it doesn't exist
    df.to_json(json_file_path, orient='records', lines=True) # Write the dataframe to a json file in the target directory
    print(json_file_path)

In [91]:
def file_converter(table_name):
    # If the files to be consumed have some common pattern in their names, we can use that pattern to filter the files to be consumed ('part-') - part-00000 is the common pattern in the files we want to consume:
    src_base_dir = '/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db'
    tgt_base_dir = '/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db_json'

    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    files = glob.glob(f'{src_base_dir}/{table_name}/part-*')

    for file in files:
        df = read_csv(file, schemas)
        file_name = file_path_list[-1]
        to_json(df, tgt_base_dir, table_name, file_name)

In [92]:
ds_name = 'orders'

In [93]:
file_converter(ds_name)

/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/data/retail_db_json/orders/part-00000
